# CDC-9 — (Deep) failure-mode tour

**Break → Detect → Fix → Prove.** The capstone of the CDC track. CDC-1…CDC-8 built the pipeline
**Postgres → Debezium (Kafka Connect) → Kafka → Spark → Iceberg**; this module steps back to the
on-call questions — *what happens when the connector restarts? are events ordered? can they
duplicate? is any of this exactly-once?* The through-line: **Kafka Connect is at-least-once with
per-key ordering, so you design for replay + idempotency, not for magic exactly-once.**

**Prerequisite:** `make cdc-up` (Postgres `:5432`, Connect REST `:8083`, Kafka `:29092`). Watch it
live in **kafka-ui** → http://localhost:8080 (topic `dbz.public.cdc9_orders`, plus internal
`connect_offsets` / `connect_status`). **Laptop-safe:** 8-row table, bounded reads, full teardown.

> **Honesty note (same stance as the SPK-2/3 OOM and KAF-5 modules).** This is a failure-mode
> *tour*. The **deterministic** parts — the connector **lifecycle (pause → resume → restart)** and
> **offset recovery** from `connect_offsets` — we **run for real** and assert on. The parts that
> need a real crash, a multi-key race, or a second runtime — **ordering across partitions,
> duplicate-on-crash, end-to-end exactly-once, Connect vs Debezium Server, slot/WAL risk** — we
> **describe precisely with correct snippets** and tie to where each is demonstrated for real
> (CDC-5, CDC-7, KAF-1/5, STR-2).

In [ ]:
from common import cdc_helpers as cdc
import json, time

NAME, TBL = "cdc9-orders", "cdc9_orders"
TOPIC = cdc.topic_name(TBL)

print("Connect REST up:", cdc.connect_up(timeout=40))
cdc.teardown(NAME, TBL)          # clean slate so the run is idempotent (delete connector/slot/table/topic)
cdc.seed_orders(TBL, n=8)
print("seeded public.cdc9_orders (8 rows); Debezium topic:", TOPIC)

## 1. Register the connector (the pipeline under test)

A standard Debezium Postgres connector (`snapshot.mode=initial`): snapshot the 8 rows as `r`
events, then stream every change. Same config shape as CDC-2 — here it's the *subject* of the
failure-mode tour, not the lesson itself.

In [ ]:
cfg = cdc.debezium_pg_config(NAME, TBL, snapshot_mode="initial")
print("register -> HTTP", cdc.register_connector(NAME, cfg)["status"])   # 201
print("state    ->", cdc.wait_for_connector(NAME, timeout=60))            # RUNNING

## 2. REAL demo — connector lifecycle & offset recovery

Kafka Connect stores each source connector's progress — for Debezium, the **WAL position (LSN)**
it has emitted up to — in the internal **`connect_offsets`** Kafka topic. So **pause** and
**restart** are *not* a reset: on resume the connector reads its stored offset and continues from
the next change. We prove it:

1. snapshot (8 `r`) + one live INSERT → one `c`;
2. **pause** the connector → status `PAUSED`;
3. INSERT 3 rows **while paused** → they sit unread in the WAL (the topic `c` count stays put);
4. **resume** → after a short decode lag, **all 4** `c` events appear — the 3 outage-time changes
   **streamed through on resume**, from the stored offset, *not* a re-snapshot.

> **Read timing.** Debezium streaming has a small decode lag, so we **wait generously, then drain
> once** (`read_cdc_events` is bounded by `max_ms`, so it always returns). Eager reads would race
> the connector — the events are coming, just not instantly.

In [ ]:
# snapshot + one streamed change. Wait, then drain ONCE (snapshot 'r' + the 'c' land together).
cdc.pg_exec("INSERT INTO public.cdc9_orders(id,customer,amount,status) VALUES (100,'live',9.90,'NEW')")
time.sleep(15)
ev = cdc.read_cdc_events(TOPIC, max_ms=12000)
print("op counts after snapshot + 1 insert:", cdc.op_counts(ev))   # {'r': 8, 'c': 1}

In [ ]:
# PAUSE via the Connect REST API, then write to Postgres while the connector is down.
code_, _ = cdc._req("PUT", f"/connectors/{NAME}/pause")
print("pause -> HTTP", code_)                                       # 202 accepted
time.sleep(4)
st = cdc.connector_status(NAME)
print("state ->", st["connector"]["state"], "| tasks:", [t["state"] for t in st["tasks"]])  # PAUSED / PAUSED

for i in (200, 201, 202):
    cdc.pg_exec(f"INSERT INTO public.cdc9_orders(id,customer,amount,status) VALUES ({i},'paused',1.00,'NEW')")
print("inserted 3 rows in Postgres WHILE paused")
time.sleep(4)
ev = cdc.read_cdc_events(TOPIC, max_ms=8000)
print("op counts while paused (c still 1 — connector consumes nothing):", cdc.op_counts(ev))

In [ ]:
# RESUME — the connector reads its stored offset from connect_offsets and catches up.
code_, _ = cdc._req("PUT", f"/connectors/{NAME}/resume")
print("resume -> HTTP", code_)                                      # 202
print("state  ->", cdc.wait_for_connector(NAME, timeout=40))         # RUNNING
time.sleep(15)                                                        # let it drain the 3 buffered changes
ev = cdc.read_cdc_events(TOPIC, max_ms=12000)
counts = cdc.op_counts(ev)
print("op counts after resume:", counts)                             # {'r': 8, 'c': 4}
print("\nOFFSET RECOVERY:", counts.get("c", 0), "create events ->",
      "the 3 changes made during the outage streamed on resume" if counts.get("c", 0) >= 4
      else "(still draining — Debezium decode lag; re-run this cell)")

## 3. REAL demo — the restart endpoint

**`POST /connectors/<n>/restart`** restarts the connector instance (the first thing you try on a
wedged task). Because offsets are durable in `connect_offsets`, a restart re-reads from the
committed LSN — **safe and non-destructive**. `?includeTasks=true&onlyFailed=false` also bounces
the task.

In [ ]:
code_, _ = cdc._req("POST", f"/connectors/{NAME}/restart")
print("restart            -> HTTP", code_)                           # 204 (no content)
time.sleep(3)
print("state after restart ->", cdc.connector_status(NAME)["connector"]["state"])  # RUNNING

code_, _ = cdc._req("POST", f"/connectors/{NAME}/restart?includeTasks=true&onlyFailed=false")
print("restart (+tasks)   -> HTTP", code_)                           # 202
time.sleep(3)
st = cdc.connector_status(NAME)
print("final state        ->", st["connector"]["state"], "| tasks:", [t["state"] for t in st["tasks"]])

## 4. Described — Connect vs Debezium Server

We deploy Debezium **inside Kafka Connect** (the brief mandates this — it mirrors most production
shops, and it's what gives us the REST lifecycle we just drove). The alternative is **Debezium
Server**: a standalone runtime that pushes change events straight to a sink (Kinesis, Pub/Sub,
Pulsar, HTTP) **without a Kafka cluster or the Connect framework**.

| | Kafka Connect + Debezium (here) | Debezium Server (standalone) |
|---|---|---|
| Runtime | Connect distributed workers | one JVM process |
| Control plane | **REST API** (pause/resume/restart, config) | config file / env, no REST control plane |
| Offsets/status | Kafka topics (`connect_offsets`, `connect_status`) | a local/offset backing store |
| Sinks | the whole Connect **sink-connector catalogue** + SMTs | a fixed set of built-in sinks |
| Cost | must run Kafka + Connect | lighter to operate |

Trade-off, not a correctness difference: Connect buys you the ecosystem and control plane at the
cost of running it; Debezium Server is leaner but gives up distribution, the sink catalogue, and
the REST control. **Both are at-least-once.**

## 5. Described — ordering guarantees (ties to KAF-1)

Debezium routes a table's events to one topic and **partitions by the primary key** (hash of the
key — exactly like KAF-1). Kafka guarantees order **within a partition**, so you get:

- ✅ **Per-key ordering** — every change to row `id=42` arrives in commit order, always.
- ❌ **No global / cross-key ordering** — changes to different keys live in different partitions
  and may interleave; **out-of-order is only ever *across* partitions, never within one**.

For a CDC mirror this is the right contract: the MERGE for a given key applies that key's changes
in order; it never needs a global order. The default key partitioner (sketch):

```text
partition = hash(primary_key) % num_partitions      # same key -> same partition -> ordered
```

## 6. Described — out-of-order / duplicate delivery (ties to KAF-5, STR-2, CDC-7)

Kafka Connect is **at-least-once**. The danger window: the connector emits a batch to Kafka, then
**crashes before committing the new offset** to `connect_offsets`. On restart it resumes from the
*last committed* offset and **re-emits** the records after it — downstream sees **duplicates**
(and, across partitions, possible reordering relative to a *different* key). No at-least-once
setting removes this. The cure is a **downstream sink that absorbs replays** — dedupe / upsert on
the primary key with an **LSN-monotonic guard** so an older replayed change can't clobber a newer
applied one. That is the CDC-7 sink:

```sql
-- CDC-7 idempotent MERGE: PK match + LSN guard makes a replay harmless
MERGE INTO iceberg_catalog.cdc.orders t
USING changes s
  ON t.id = s.id
WHEN MATCHED AND s.op = 'd'                       THEN DELETE
WHEN MATCHED AND s.source_lsn > t.source_lsn      THEN UPDATE SET *   -- only newer LSN wins
WHEN NOT MATCHED AND s.op != 'd'                  THEN INSERT *
```

The KAF-5 signature of duplicates — **offsets produced ≫ distinct keys** — is how you *detect*
this in the sink's input.

## 7. Described — end-to-end exactly-once reasoning (the senior point)

There is **no magic EOS** across `PG → Kafka → Spark → Iceberg`; every hop is at-least-once. You
engineer **effectively-once** by stacking idempotency at each layer:

| Layer | Mechanism | Module |
|-------|-----------|--------|
| Connect → Kafka | at-least-once + per-key order (offsets in `connect_offsets`) | **CDC-9** (this notebook) |
| Kafka → Spark | Structured Streaming **checkpoint** = durable offset store; safe restart | **STR-2** |
| Spark consumer | `isolation.level=read_committed` (ignore aborted txns) | **KAF-5** |
| Spark → Iceberg | idempotent **MERGE** keyed by PK + **LSN-monotonic guard** | **CDC-7** |

The combination is the answer: *at-least-once delivery + an LSN-guarded PK MERGE + checkpointed
offsets + read_committed* ⇒ each source change affects the Iceberg mirror **once**, even though
the wire may carry it twice. Don't hand-roll Kafka EOS transactions across this boundary — lean on
the checkpoint (STR-2).

## 8. Described — slot / WAL risk on a prolonged outage (ties to CDC-5)

Our pause was seconds, so it was safe. **Left paused (or FAILED) for hours**, the connector stops
advancing its slot's `restart_lsn`, so **Postgres retains every WAL segment since that LSN** —
unbounded disk growth (the CDC-5 pathology). `list_slots()` exposes `retained_bytes` per slot —
an inactive slot with a rising value is the alarm. Fix in CDC-5: monitor slot age, cap with
`max_slot_wal_keep_size`, keep consumers healthy (or drop a dead slot).

In [ ]:
# Our connector's slot is ACTIVE and advancing (healthy). A paused/failed connector for hours
# would show this slot inactive with retained_bytes climbing -> the CDC-5 incident.
for s in cdc.list_slots():
    print(s)

## 9. Takeaways & "in real production…"

- **Per-key ordering is the contract.** Build downstream logic needing only per-key order; never
  assume global order across a CDC topic's partitions.
- **At-least-once + idempotency beats chasing once-only delivery.** Replays you dedupe; lost
  changes you can't. The robust CDC sink is an LSN-guarded, PK-keyed MERGE (CDC-7).
- **There is no end-to-end EOS switch** — effectively-once is *assembled*: Connect offsets +
  Streaming checkpoint + `read_committed` + idempotent MERGE. Know which layer gives which.
- **Connector lifecycle is safe by design** (offsets are durable) — but a *prolonged* outage is a
  **WAL-retention incident** (CDC-5). Monitor task state, lag, and slot `retained_bytes` together.
- **Connect vs Debezium Server** is a deployment trade-off (ecosystem & control plane vs weight),
  not a correctness one — both are CDC, both at-least-once.

## Teardown

In [ ]:
cdc.teardown(NAME, TBL)   # delete connector, drop slot, drop table, delete topic
print("slots now:", cdc.list_slots())